# Strategic / Portfolio Risk & ESG Engine — full run

Concentration (HHI, Gini) · rating migration · ESG scoring · NGFS climate stress.

Runs end to end with no token and no download: the portfolio is built
deterministically. **Runtime → Run all.**

In [ ]:
import os, sys
if not os.path.exists('src/concentration.py'):
    !git clone https://github.com/ArjiNasim/strategic-esg-risk-engine.git
    %cd strategic-esg-risk-engine
!pip -q install matplotlib pyarrow
sys.path.insert(0, os.getcwd())

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from src import portfolio, concentration, migration, esg, climate
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

## 1. The portfolio

In [ ]:
df = portfolio.build_portfolio(n=200, seed=42)
print(portfolio.portfolio_summary(df))
df.head(10)

In [ ]:
df.groupby('sector')['ead'].sum().sort_values().plot(
    kind='barh', figsize=(8,4), color='#164e63', title='Exposure by sector (M)')
plt.tight_layout(); plt.show()

## 2. Concentration

In [ ]:
cr = concentration.concentration_report(df)
print(f"HHI name  : {cr['hhi_name']:.4f}  (effective names {cr['effective_number_names']:.0f})")
print(f"Top-10    : {cr['top10_share']:.1%}   Gini {cr['gini']:.3f}")
print(f"HHI sector/geo: {cr['hhi_sector']:.3f} / {cr['hhi_geography']:.3f}")
cr['by_sector']

## 3. Rating migration

In [ ]:
mr = migration.migration_report(df)
print(f"Portfolio 1Y PD: {mr['portfolio_pd_1y']:.2%}")
print(f"Expected EAD default (1Y): {mr['expected_ead_default_1y']:,.0f}M")
mr['projected_1y']

In [ ]:
proj = mr['projected_1y']; proj = proj[proj['rating']!='D']
x = np.arange(len(proj)); w=0.38
plt.figure(figsize=(8,3.5))
plt.bar(x-w/2, proj['current']*100, w, label='current', color='#164e63')
plt.bar(x+w/2, proj['projected']*100, w, label='projected 1Y', color='#b8860b')
plt.xticks(x, proj['rating']); plt.ylabel('% of EAD'); plt.legend()
plt.title('Rating distribution: current vs projected'); plt.tight_layout(); plt.show()
mr['multi_year_pd']

## 4. ESG scoring

In [ ]:
er = esg.esg_report(df)
print(f"Weighted ESG score: {er['weighted_esg_score']:.1f} ({er['weighted_band']})")
print(f"Sensitive-sector share: {er['sensitive_sector_share']:.1%}")
er['by_band']

In [ ]:
er['worst_contributors']

## 5. Climate stress (NGFS)

In [ ]:
clr = climate.climate_report(df)
clr['scenarios'].round(1)

In [ ]:
sc = clr['scenarios']
x = np.arange(len(sc))
plt.figure(figsize=(8,4))
plt.bar(x, sc['transition_loss'], color='#164e63', label='transition')
plt.bar(x, sc['physical_loss'], bottom=sc['transition_loss'], color='#b8860b', label='physical')
plt.xticks(x, [s.split(' (')[0] for s in sc['scenario']]); plt.ylabel('loss (M)'); plt.legend()
plt.title('Climate stress loss by NGFS scenario'); plt.tight_layout(); plt.show()
print('worst:', clr['worst_scenario'], f"{clr['worst_loss']:,.0f}M")

## 6. Save artefact

In [ ]:
import os; os.makedirs('data', exist_ok=True)
df.to_parquet('data/portfolio.parquet')
print('saved data/portfolio.parquet')
print('\nStrategic / ESG risk engine run complete.')